In [1]:
import trafilatura
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
import tempfile
import requests
import json
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import re
import hashlib
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# Configuration
KEYWORDS = ["licenta", "master", "doctorat", "calendar", "taxe", "acte", "admitere", "candidati"]
BAD_KEYWORDS = ["anunt", "finalizare", "anunturi", "xls", "teze", "docx", "plan", "pdf", "uploads", "download", "postuniversitare", "2000", "2001", "2002", "2003", "2004", "2005", "2006", "2007", "2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]
START_URLS = ["https://www.tuiasi.ro/admitere", 
              "https://ac.tuiasi.ro/admitere", 
              "https://arh.tuiasi.ro/admitere-2025/", 
              "https://etti.tuiasi.ro/admitere", 
              "https://icpm.tuiasi.ro/admitere/", 
              "https://ci.tuiasi.ro/admitere/", 
              "https://cmmi.tuiasi.ro/admitere/", 
              "https://hgim.tuiasi.ro/admitere/",  
              "https://ieeia.tuiasi.ro/admitere/", 
              "https://mec.tuiasi.ro/admitere/", 
              "https://sim.tuiasi.ro/admitere/", 
              "https://dima.tuiasi.ro/admitere/"]    


In [9]:
def _url_path_tokens(parsed_url):
    return [t for t in re.split(r"[^a-z0-9]+", parsed_url.path.lower()) if t]


def get_page_links(url):
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return []

        soup = BeautifulSoup(downloaded, "html.parser")
        absolute_links = set()
        base_domain = urlparse(url).netloc

        for a_tag in soup.find_all("a", href=True):
            href = a_tag["href"].lower()
            full_url = urljoin(url, href)

            parsed_url = urlparse(full_url)
            clean_link = f"{parsed_url.scheme}://{parsed_url.netloc}{parsed_url.path}".rstrip("/")

            # 1. Domain Check: Must stay on the same faculty/university domain
            if base_domain in parsed_url.netloc and parsed_url.scheme in ["http", "https"]:
                tokens = _url_path_tokens(parsed_url)

                # 2. Keyword Check (whole-token): only match complete tokens in the PATH
                has_good_word = any(word in tokens for word in KEYWORDS)
                has_bad_word = any(word in tokens for word in BAD_KEYWORDS)

                if has_good_word and not has_bad_word:
                    absolute_links.add(clean_link)

        return list(absolute_links)
    except Exception as e:
        print(f"⚠️ Error fetching links from {url}: {e}")
        return []

In [10]:
queue = [(url, 0) for url in START_URLS]
visited = set()
all_discovered_links = set()
MAX_DEPTH = 5

while queue:
    current_url, depth = queue.pop(0)
    
    if current_url in visited:
        continue
    
    visited.add(current_url)
    all_discovered_links.add(current_url)
    
    prefix = "  " * depth + "└─"
    print(f"{prefix} [{depth}] Visiting: {current_url}")
    
    if depth < MAX_DEPTH:
        sublinks = get_page_links(current_url)
        
        new_links_found = 0
        for link in sublinks:
            if link not in visited:
                queue.append((link, depth + 1))
                new_links_found += 1
        
        if new_links_found > 0:
            pass 

with open("urls.txt", "w", encoding="utf-8") as f:
    for link in sorted(all_discovered_links):
        f.write(f"{link}\n")

print("💾 List saved in 'urls.txt'.")

└─ [0] Visiting: https://www.tuiasi.ro/admitere
└─ [0] Visiting: https://ac.tuiasi.ro/admitere
└─ [0] Visiting: https://arh.tuiasi.ro/admitere-2025/
└─ [0] Visiting: https://etti.tuiasi.ro/admitere
└─ [0] Visiting: https://icpm.tuiasi.ro/admitere/
└─ [0] Visiting: https://ci.tuiasi.ro/admitere/
└─ [0] Visiting: https://cmmi.tuiasi.ro/admitere/
└─ [0] Visiting: https://hgim.tuiasi.ro/admitere/
└─ [0] Visiting: https://ieeia.tuiasi.ro/admitere/
└─ [0] Visiting: https://mec.tuiasi.ro/admitere/
└─ [0] Visiting: https://sim.tuiasi.ro/admitere/
└─ [0] Visiting: https://dima.tuiasi.ro/admitere/
  └─ [1] Visiting: https://www.tuiasi.ro/licenta/taxe-si-finantare
  └─ [1] Visiting: https://www.tuiasi.ro/doctorat/documente-utile
  └─ [1] Visiting: https://www.tuiasi.ro/acte-de-studii
  └─ [1] Visiting: https://www.tuiasi.ro/doctorat
  └─ [1] Visiting: https://www.tuiasi.ro/masterat/taxe-si-finantare
  └─ [1] Visiting: https://www.tuiasi.ro/admitere/doctorat
  └─ [1] Visiting: https://www.tuiasi.r

In [11]:
def get_faculty_tag(url):
    domain = urlparse(url).netloc
    parts = domain.split('.')
    
    faculty_map = {
        "ac": "Automatică și Calculatoare",
        "arh": "Arhitectură",
        "icpm": "Inginerie Chimică și Protecția Mediului",
        "ci": "Construcții și Instalații",
        "dima": "Design Industrial și Managementul Afacerilor",
        "ieeia": "Inginerie Electrică, Energetică și Informatică Aplicată",
        "etti": "Electronică, Telecomunicații și Tehnologii Informaționale",
        "hgim": "Hidrotehnică, Geodezie și Ingineria Mediului",
        "mec": "Mecanică",
        "sim": "Știința și Ingineria Materialelor",
        "cmmi": "Construcții de Mașini și Management Industrial"
    }

    if len(parts) >= 3 and parts[-2] == "tuiasi":
        abbr = parts[0].lower()
        if abbr in faculty_map:
            return f"{faculty_map[abbr]} ({abbr.upper()})"
        else:
            return "TUIASI" 
            
    return "TUIASI"

def get_url_tags(url):
    url_lower = url.lower()
    base_keywords = ["licenta", "master", "doctorat", "candidati", "taxe", "studii", "studies"]
    found_tags = [word for word in base_keywords if word in url_lower]
    if "/en/" in url_lower:
        found_tags.append("english")    
    return found_tags

def get_all_metadata_tags(url):
    faculty = get_faculty_tag(url)
    category_tags = get_url_tags(url)
    return [faculty] + category_tags

print(get_all_metadata_tags("https://ac.tuiasi.ro/en/students/didactic/calendar"))

['Automatică și Calculatoare (AC)', 'english']


In [12]:
def fetch_and_clean(url):
    downloaded = trafilatura.fetch_url(url)
    if not downloaded:
        return None  
        
    # Parse with BeautifulSoup first
    soup = BeautifulSoup(downloaded, 'html.parser')
    
    # Destroy common boilerplate tags before Trafilatura sees them
    for bad_tag in soup(['nav', 'header', 'footer', 'aside', 'script', 'style']):
        bad_tag.decompose()
        
    # Also find tags with common classes/ids used for menus
    for menu in soup.find_all(['div', 'ul'], class_=['menu', 'navbar', 'sidebar', 'widget']):
        menu.decompose()
        
    # Pass the cleaned HTML string to Trafilatura
    cleaned_html = str(soup)
    return trafilatura.extract(
        cleaned_html, 
        include_formatting=True, 
        include_links=False, 
        favor_precision=True
    )

def clean_extracted_text(text):
    if not text:
        return ""
    # Remove lines that are just dashes or empty bullets
    text = re.sub(r'^\s*-\s*$', '', text, flags=re.MULTILINE)
    # Remove repetitive "MENUMENU" artifacts
    text = text.replace("MENUMENU", "")
    # Remove multiple empty blank lines
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

def get_page_content(url):
    return clean_extracted_text(fetch_and_clean(url))

In [28]:
OUTPUT_FILE = "database_documents.json"
PARENT_STORE_DIR = "parent_store"
CHILD_CHUNK_SIZE = 500
CHILD_CHUNK_OVERLAP = 100

os.makedirs(PARENT_STORE_DIR, exist_ok=True)

# 1. Parent Splitter (Splits by logical Markdown headers)
headers_to_split_on = [
    ("#", "H1"), 
    ("##", "H2"), 
    ("###", "H3")
]
parent_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 2. Child Splitter (Splits the parents into small, searchable pieces)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=CHILD_CHUNK_OVERLAP
)

# --- MAIN PIPELINE ---
if not os.path.exists("urls.txt"):
    print("❌ Error: 'urls.txt' not found.")
    exit()

with open("urls.txt", "r", encoding="utf-8") as f:
    urls = [line.strip() for line in f.readlines() if line.strip()]

print(f"🚀 Starting processing for {len(urls)} URLs...")
print(f"⚙️ Chunk Settings: Parents by Header, Children {CHILD_CHUNK_SIZE} chars.")


for url in urls:
    try:
        raw_text = get_page_content(url)

        if not raw_text or len(raw_text.strip()) < 100:
            print(f"⚠️ Ignored (Content too short or empty): {url}")
            continue
            
        tags = get_all_metadata_tags(url) # Call your existing metadata function
        
        # 1. SPLIT INTO PARENTS
        parent_chunks = parent_splitter.split_text(raw_text)
        
        # Fallback: If the page has no headers, treat the whole page as one Parent
        if not parent_chunks:
            parent_chunks = [Document(page_content=raw_text, metadata={})]

        total_children_for_url = 0

        # 2. PROCESS EACH PARENT
        for i, p_chunk in enumerate(parent_chunks):
            # Generate a unique, safe ID for this parent (URL hash + index)
            url_hash = hashlib.md5(url.encode('utf-8')).hexdigest()[:8]
            parent_id = f"parent_{url_hash}_{i}"
            
            # Extragem headerele din metadata (ex: {"H2": "Acte necesare", "H3": "Master"})
            # Le unim într-un singur string. Dacă nu are headere, lăsăm gol.
            headers_dict = p_chunk.metadata
            section_path = " > ".join(headers_dict.values()) if headers_dict else "Secțiune Generală"
            
            # --- SAVE PARENT TO LOCAL DISK ---
            parent_data = {
                "parent_id": parent_id,
                "source": url,
                "tags": tags,
                "headers": headers_dict, 
                "page_content": p_chunk.page_content
            }
            parent_filepath = os.path.join(PARENT_STORE_DIR, f"{parent_id}.json")
            with open(parent_filepath, "w", encoding="utf-8") as pf:
                json.dump(parent_data, pf, ensure_ascii=False, indent=2)

            # 3. SPLIT PARENT INTO CHILDREN
            p_chunk.metadata.update({"source": url, "parent_id": parent_id, "tags": tags})
            child_chunks = child_splitter.split_documents([p_chunk])
            
            # --- SAVE CHILDREN TO DATABASE JSON ---
            with open(OUTPUT_FILE, "a", encoding="utf-8") as file_out:
                for child in child_chunks:
                    # ACUM INJECTĂM 'section_path' DIRECT ÎN TEXTUL COPILULUI!
                    tag_header = (
                        f"TAGURI: {', '.join(tags)}\n"
                        f"SECȚIUNE: {section_path}\n"  # <-- Aici e magia!
                        f"SOURCE: {url}\n"
                        f"PARENT_ID: {parent_id}\n\n"
                    )
                    final_content = tag_header + child.page_content

                    json_obj = {
                        "page_content": final_content,
                        "source": url,
                        "tags": tags,
                        "parent_id": parent_id
                    }
                    file_out.write(json.dumps(json_obj, ensure_ascii=False) + "\n")
            
            total_children_for_url += len(child_chunks)

        print(f"   ✅ Processed {len(parent_chunks)} Parents and {total_children_for_url} Children for: {url}")

    except Exception as e:
        print(f"⚠️ Error processing {url}: {e}")

print(f"\n✨ Done! Child chunks saved to '{OUTPUT_FILE}'. Parents saved to '{PARENT_STORE_DIR}/'.")

🚀 Starting processing for 162 URLs...
⚙️ Chunk Settings: Parents by Header, Children 500 chars.
⚠️ Ignored (Content too short or empty): http://www.ci.tuiasi.ro/ro/admitere/admitere-studii-licenta/studii-universitare-de-licenta-admitere
⚠️ Ignored (Content too short or empty): http://www.ci.tuiasi.ro/ro/admitere/admitere-studii-master/studii-universitare-de-master-admitere
   ✅ Processed 1 Parents and 1 Children for: https://ac.tuiasi.ro/admitere
   ✅ Processed 3 Parents and 5 Children for: https://ac.tuiasi.ro/admitere/continuare-studii
   ✅ Processed 12 Parents and 48 Children for: https://ac.tuiasi.ro/admitere/doctorat
   ✅ Processed 13 Parents and 43 Children for: https://ac.tuiasi.ro/admitere/licenta
   ✅ Processed 8 Parents and 27 Children for: https://ac.tuiasi.ro/admitere/masterat
   ✅ Processed 1 Parents and 1 Children for: https://ac.tuiasi.ro/en/students/didactic/calendar
   ✅ Processed 4 Parents and 6 Children for: https://ac.tuiasi.ro/en/studies/masters-degree/ai-master-ar

In [2]:
INPUT_FILE = "database_documents.json"
DB_DIR = "./chroma_db"

def load_documents_from_jsonl(file_path):
    """Reads documents from the previously generated JSONL file."""
    docs = []
    if not os.path.exists(file_path):
        print(f"❌ Error: File {file_path} not found.")
        return docs

    print(f"📂 Reading documents from {file_path}...")
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                docs.append(
                    Document(
                        page_content=data.get("page_content", ""),
                        metadata={
                            "tags": data.get("tags", []),
                            "source": data.get("source"),
                        },
                    )
                )

    return docs

def build_vector_database():
    """Processes chunks and stores them in the ChromaDB vector database."""
    documents = load_documents_from_jsonl(INPUT_FILE)
    
    if not documents:
        print("⚠️ No documents found to populate the database.")
        return

    print(f"✅ Loaded {len(documents)} fragments (chunks).")
    
    # Using a high-performance multilingual embedding model for Romanian/English
    print("🧠 Downloading/Loading embedding model (paraphrase-multilingual)...")
    embeddings_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

    print("🏗️ Building ChromaDB vector database. This might take a few moments...")
    
    # Initialize Chroma and save the data to the local directory
    vector_db = Chroma.from_documents(
        documents=documents,
        embedding=embeddings_model,
        persist_directory=DB_DIR
    )
    
    print(f"✨ Done! The database has been successfully saved to the '{DB_DIR}' directory.")
    return vector_db

db = build_vector_database()

📂 Reading documents from database_documents.json...
✅ Loaded 1832 fragments (chunks).
🧠 Downloading/Loading embedding model (paraphrase-multilingual)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏗️ Building ChromaDB vector database. This might take a few moments...
✨ Done! The database has been successfully saved to the './chroma_db' directory.


In [5]:
import re

# 1. Define the mappings for Faculty Tags
FACULTY_MAP = {
    "ac": "Automatică și Calculatoare (AC)",
    "automatica": "Automatică și Calculatoare (AC)",
    "calculatoare": "Automatică și Calculatoare (AC)",
    "arh" : "Arhitectură (ARH)",
    "arhi" : "Arhitectură (ARH)",
    "arhitectura" : "Arhitectură (ARH)",
    "arhitect" : "Arhitectură (ARH)",
    "icpm": "Inginerie Chimică și Protecția Mediului (ICPM)",
    "chimie": "Inginerie Chimică și Protecția Mediului (ICPM)",
    "inginerie chimica": "Inginerie Chimică și Protecția Mediului (ICPM)",
    "ci": "Construcții și Instalații (CI)",
    "constructii": "Construcții și Instalații (CI)",
    "dima": "Design Industrial și Managementul Afacerilor (DIMA)",
    "design": "Design Industrial și Managementul Afacerilor (DIMA)",
    "management": "Design Industrial și Managementul Afacerilor (DIMA)",
    "ieeia": "Inginerie Electrică, Energetică și Informatică Aplicată (IEEIA)",
    "electrica": "Inginerie Electrică, Energetică și Informatică Aplicată (IEEIA)",
    "etti": "Electronică, telecomunicații și tehnologia informației (ETTI)",
    "electronica": "Electronică, telecomunicații și tehnologia informației (ETTI)",
    "electrotehnica": "Electronică, telecomunicații și tehnologia informației (ETTI)",
    "hgim": "Hidrotehnică, Geodezie și Ingineria Mediului (HGIM)",
    "hidro": "Hidrotehnică, Geodezie și Ingineria Mediului (HGIM)",
    "hidrotehnica": "Hidrotehnică, Geodezie și Ingineria Mediului (HGIM)",
    "geodezie": "Hidrotehnică, Geodezie și Ingineria Mediului (HGIM)",
    "geologie": "Hidrotehnică, Geodezie și Ingineria Mediului (HGIM)",
    "mec": "Mecanică (MEC)",
    "mecanica": "Mecanică (MEC)",
    "sim": "Știința și Ingineria Materialelor (SIM)",
    "stiinta": "Știința și Ingineria Materialelor  (SIM)",
    "materiale": "Știința și Ingineria Materialelor  (SIM)",
    "cmmi": "Construcții de Mașini și Management Industrial (CMMI)",
    "masini": "Construcții de Mașini și Management Industrial (CMMI)"
    
}

# 2. Define the mappings for Secondary Tags
SECONDARY_MAP = {
    "studiu": "studii",
    "studii": "studii",
    "studia": "studii",
    "taxa": "taxe",
    "taxe": "taxe",
    "plata": "taxe",
    "bani": "taxe",
    "acte": "acte",
    "documente": "acte",
    "dosar": "acte",
    "master": "master",
    "masterat": "master",
    "licenta": "licenta",
    "candidati" : "canditati"
}

def extract_tags_from_query(query: str):
    """Scans the query and returns matching faculty and secondary tags."""
    clean_query = re.sub(r'[^\w\s]', '', query.lower())
    words = clean_query.split()
    
    found_faculty = None
    found_secondary = []
    
    for word in words:
        if word in FACULTY_MAP and not found_faculty:
            found_faculty = FACULTY_MAP[word]
        if word in SECONDARY_MAP:
            tag = SECONDARY_MAP[word]
            if tag not in found_secondary:
                found_secondary.append(tag)
                
    # Acum returnăm DOUĂ valori!
    return found_faculty, found_secondary

def filtered_docs(query: str, limit: int = 5):
    """Filtrare inteligentă cu Fallback (de la strict la relaxat)"""
    # Aici despachetăm corect ambele valori
    faculty_tag, secondary_tags = extract_tags_from_query(query)
    
    # ETAPA 1: Filtru STRICT (Facultate + Un tag secundar găsit, ex: 'master' sau 'acte')
    if faculty_tag and secondary_tags:
        # Încercăm să filtrăm după fiecare tag secundar găsit, pe rând
        for sec_tag in secondary_tags:
            strict_filter = {
                "$and": [
                    {"tags": {"$contains": faculty_tag}},
                    {"tags": {"$contains": sec_tag}}
                ]
            }
            try:
                results = db.similarity_search(query, k=limit, filter=strict_filter)
                if results:
                    return results # Am găsit un meci perfect!
            except Exception:
                continue # Dacă crapă (ex: tag-ul nu există în baza de date), trecem la următorul

    # ETAPA 2: Fallback la Facultate (Dacă filtrul strict nu a adus nimic)
    if faculty_tag:
        faculty_only_filter = {"tags": {"$contains": faculty_tag}}
        try:
            results = db.similarity_search(query, k=limit, filter=faculty_only_filter)
            if results:
                return results 
        except Exception:
             pass 
             
    # ETAPA 3: Fallback Absolut (Niciun filtru)
    return db.similarity_search(query, k=limit)

In [20]:
from langchain_core.tools import tool

@tool
def search_child_chunks(query: str, limit: int = 3) -> str:
    """Search the vector database for the most relevant specific pieces of admission data.
    Use this tool FIRST to find specific keywords, deadlines, fees, or short facts.
    Returns short chunks of text along with a 'Parent ID'.
    """
    try:
        # Assuming your ChromaDB instance is named 'vector_db' (from your notebook)
        results = db.filtered_docs(query, limit)
        
        if not results:
            return "NO_RELEVANT_CHUNKS_FOUND"

        formatted_results = []
        for doc in results:
            parent_id = doc.metadata.get('parent_id', 'UNKNOWN')
            source = doc.metadata.get('source', 'UNKNOWN')
            content = doc.page_content.strip()
            
            # Formatting it cleanly so the LLM can easily read the Parent ID
            formatted_results.append(
                f"Parent ID: {parent_id}\nSource: {source}\nContent: {content}"
            )
        
        return "\n\n---\n\n".join(formatted_results)

    except Exception as e:
        return f"RETRIEVAL_ERROR: {str(e)}"


# --- TOOL 2: The Context Fetcher ---
@tool
def retrieve_parent_chunks(parent_id: str) -> str:
    """Retrieve the full, detailed parent section by its exact ID.
    Use this tool SECOND, ONLY IF the information from 'search_child_chunks' seems cut off, 
    incomplete, or if you need the surrounding context (like a full table of fees or a complete list of documents).
    Input MUST be the exact 'Parent ID' string returned by the search tool.
    """
    # Ensure safe filename handling
    file_name = parent_id if parent_id.lower().endswith(".json") else f"{parent_id}.json"
    path = os.path.join(PARENT_STORE_DIR, file_name)

    if not os.path.exists(path):
        return f"NO_PARENT_DOCUMENT_FOUND_FOR_ID: {parent_id}"

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        return (
            f"Parent ID: {parent_id}\n"
            f"Source: {data.get('source', 'unknown')}\n"
            f"FULL CONTENT:\n{data.get('page_content', '').strip()}"
        )
    except Exception as e:
         return f"ERROR_READING_PARENT: {str(e)}"

In [7]:
# --- TEST SCRIPT ---
test_query = "Care sunt actele necesare la dosar pentru master la calculatoare?"

print(f"🔍 Testing Query: '{test_query}'")
print("-" * 50)

# De data asta despachetăm corect ambele!
faculty_tag, secondary_tags = extract_tags_from_query(test_query)

print("1️⃣ TAG EXTRACTION RESULT:")
print(f"   🏛️ Faculty Tag: {faculty_tag}")
print(f"   🏷️ Secondary Tags: {secondary_tags}")
print("-" * 50)

print("2️⃣ DATABASE RETRIEVAL RESULT:")
try:
    retrieved_docs = filtered_docs(test_query, limit=5)
    
    if not retrieved_docs:
        print("   ❌ No documents found in ChromaDB.")
    else:
        for i, doc in enumerate(retrieved_docs, 1):
            parent_id = doc.metadata.get('parent_id', 'Unknown')
            doc_tags = doc.metadata.get('tags', [])
            print(f"\n--- 📄 Document {i} ---")
            print(f"   🔗 Parent ID : {parent_id}")
            print(f"   🔖 Doc Tags  : {doc_tags}")
            print(f"   📝 Content   : {doc.page_content}")

except Exception as e:
    print(f"⚠️ Error during retrieval: {e}")

🔍 Testing Query: 'Care sunt actele necesare la dosar pentru master la calculatoare?'
--------------------------------------------------
1️⃣ TAG EXTRACTION RESULT:
   🏛️ Faculty Tag: Automatică și Calculatoare (AC)
   🏷️ Secondary Tags: ['acte', 'master']
--------------------------------------------------
2️⃣ DATABASE RETRIEVAL RESULT:

--- 📄 Document 1 ---
   🔗 Parent ID : Unknown
   🔖 Doc Tags  : ['Automatică și Calculatoare (AC)', 'master', 'studies', 'english']
   📝 Content   : TAGURI: Automatică și Calculatoare (AC), master, studies, english
SECȚIUNE: Secțiune Generală
SOURCE: https://ac.tuiasi.ro/en/studies/masters-degree/ec-master-embedded-computers
PARENT_ID: parent_dbfeb97b_0

Extracts from the discipline sheets for the subjects studied in the field of Computer Science and Information Technology.

--- 📄 Document 2 ---
   🔗 Parent ID : Unknown
   🔖 Doc Tags  : ['Automatică și Calculatoare (AC)', 'master']
   📝 Content   : TAGURI: Automatică și Calculatoare (AC), master
SECȚIUNE: